## 2.1 Go 的基本类型体系

## 2.2 Slice、Map 与 Go 集合体系

### 2.2.1 slice 底层原理

### 2.2.2 map 底层原理

### 2.2.3 sync.Map

## 2.3 string、[]byte、rune

### 2.3.1 string 底层结构

这一章主要解决四个问题：

第一，Go 的类型系统到底怎么理解。
面试里很多人会说 Go 是值传递，但一遇到 slice、map、channel 就开始混乱。真正要讲清楚的是：Go 语言层面只有值传递，但某些值本身是一个“描述符”或者“引用语义的头部结构”，所以传过去以后仍然能影响底层数据。

第二，slice 为什么是 Go 里最常考的数据结构。
Java 里常问 ArrayList 扩容、LinkedList 使用场景、HashMap 底层；Go 里对应最常问的就是 slice 的底层数组、len、cap、append 扩容、共享底层数组、截取内存泄漏、删除元素这些问题。

第三，map 为什么不是线程安全的。
Go map 看起来简单，但底层涉及 bucket、overflow bucket、hash 定位、扩容、渐进迁移、随机遍历、并发读写 panic。面试里如果能讲到这些，基本就能体现出基础深度。

第四，string 为什么不是“字符数组”。
Go 的 string 本质是只读字节序列，不是 rune 数组。len 统计的是字节数，不是字符数。下标访问拿到的是 byte，range 遍历拿到的是 rune。这是 Go 面试里非常容易答错的地方。

# 一、Go 类型系统

## 【文本块 2】Q：Go 里有哪些常见类型？怎么分类？

Go 的类型可以先分成几大类：

第一类是基本类型，比如：

* bool
* int、int8、int16、int32、int64
* uint、uint8、uint16、uint32、uint64、uintptr
* float32、float64
* complex64、complex128
* byte
* rune
* string

其中 byte 本质上是 uint8 的别名，常用于表示原始字节；rune 本质上是 int32 的别名，常用于表示 Unicode 字符。

第二类是复合类型，比如：

* array
* slice
* map
* struct
* pointer
* function
* channel
* interface

第三类可以从语义上理解为“值语义类型”和“引用语义类型”。

值语义类型比较好理解，比如 int、bool、array、struct。赋值或者传参时，会复制整个值本身。

引用语义类型包括 slice、map、channel、function、interface 中装的某些动态值等。严格来说，Go 语言层面仍然是值传递，但这些值内部通常包含指向底层数据结构的指针，所以复制的是描述符，多个变量仍然可能指向同一份底层数据。

面试里最稳的说法是：

> Go 只有值传递，没有 Java/C++ 那种意义上的引用传递。但 slice、map、channel 这类类型的值本身包含指向底层结构的指针，所以传参时虽然复制了这个头部值，依然能修改底层数据。

---

## 【代码块 1】值类型传参：修改副本不影响原值

```go
import "fmt"

func changeInt(x int) {
    x = 100
}

a := 10
changeInt(a)
fmt.Println(a)
```

---

## 【文本块 3】代码解释

这里输出仍然是 10。

因为 int 是普通值类型，传参时会复制一份 x。函数内部修改的是副本，不会影响外面的 a。

这就是 Go 里“值传递”的最直观表现。

---

## 【代码块 2】指针传参：通过地址修改原值

```go
import "fmt"

func changeIntByPointer(x *int) {
    *x = 100
}

a := 10
changeIntByPointer(&a)
fmt.Println(a)
```

---

## 【文本块 4】代码解释

这里输出是 100。

因为传入的是 a 的地址，虽然指针本身也是值传递，但复制后的指针仍然指向同一个变量。通过 `*x` 修改时，改的是地址背后的数据。

所以 Go 里经常说：

> 所有传参都是值传递，指针传参也是复制指针值，只不过复制后的指针仍然指向同一块内存。

---

## 【文本块 5】追问：Go 是值传递还是引用传递？

标准回答：

Go 是值传递。

无论传 int、struct、slice、map、channel、pointer，本质上都是把变量的值复制一份传给函数。

但不同类型的“值”内容不一样。

int 的值就是一个整数本身。
struct 的值就是整个结构体内容。
pointer 的值是一个地址。
slice 的值是一个 slice header，里面包含底层数组指针、长度和容量。
map 的值是一个指向 runtime.hmap 的指针。
channel 的值也是指向 runtime.hchan 的指针。

所以当我们传 slice、map、channel 时，函数拿到的是头部结构或指针的副本，但它们仍然指向同一份底层数据，因此能影响底层内容。

面试里可以这样总结：

> Go 语言层面永远是值传递。所谓引用效果，是因为某些类型的值内部保存了指向底层数据结构的指针。

---

# 二、new 和 make

## 【文本块 6】Q：new 和 make 有什么区别？

这是 Go 基础面试里非常高频的问题。

new 和 make 都可以用来分配内存，但它们的语义完全不同。

new 用来为任意类型分配一块零值内存，并返回指针。

例如：

```go
p := new(int)
```

这会分配一个 int，默认值是 0，并返回 `*int`。

make 只能用于 slice、map、channel 这三种类型。

它的作用不是简单返回指针，而是完成底层运行时结构的初始化，返回类型本身。

例如：

```go
s := make([]int, 0, 10)
m := make(map[string]int)
ch := make(chan int, 10)
```

这里返回的分别是 `[]int`、`map[string]int`、`chan int`，不是指针。

为什么 slice、map、channel 需要 make？

因为它们不是简单的一块零值内存就能直接正常工作的类型，它们底层需要 runtime 初始化内部结构。

比如 map 的零值是 nil map，可以读，但不能写。必须 make 初始化后才能写入。

---

## 【代码块 3】new 分配零值并返回指针

```go
import "fmt"

p := new(int)

fmt.Println(*p)

*p = 42

fmt.Println(*p)
```

---

## 【代码块 4】make 初始化 map

```go
import "fmt"

var m1 map[string]int
fmt.Println(m1 == nil)

// 下面这行如果执行，会 panic：assignment to entry in nil map
// m1["a"] = 1

m2 := make(map[string]int)
m2["a"] = 1

fmt.Println(m2)
```

---

## 【文本块 7】追问：为什么 nil map 可以读，不能写？

nil map 没有初始化底层 hmap 结构。

读取一个不存在的 key，本身可以返回 value 类型的零值，所以读 nil map 是安全的。

但写入时，runtime 需要找到 bucket、可能还要分配 bucket、更新计数等。如果 map 本身还没有初始化底层结构，就没有地方可以写，所以会 panic。

因此工程里一定要注意：

```go
var m map[string]int
```

这只是声明了一个 nil map。

真正可写需要：

```go
m = make(map[string]int)
```

---

# 三、array 和 slice

## 【文本块 8】Q：array 和 slice 有什么区别？

Go 里的 array 是定长数组，长度是类型的一部分。

比如：

```go
var a [3]int
var b [4]int
```

`[3]int` 和 `[4]int` 是两个不同类型。

array 是值类型，赋值和传参会复制整个数组。

slice 是动态数组的抽象，它本身不是数组，而是一个描述符，底层指向某个数组。slice 的长度可以变化，append 时可能触发扩容。

面试里可以这样回答：

> array 是定长值类型，长度属于类型的一部分；slice 是对底层数组的动态视图，本身包含指针、长度、容量，使用更灵活，也是 Go 项目中最常用的序列结构。

---

## 【代码块 5】array 赋值会复制整个数组

```go
import "fmt"

a := [3]int{1, 2, 3}
b := a

b[0] = 100

fmt.Println("a:", a)
fmt.Println("b:", b)
```

---

## 【文本块 9】代码解释

输出中，a 仍然是 `[1 2 3]`，b 是 `[100 2 3]`。

因为数组赋值时复制了整个数组，a 和 b 是两份独立的数据。

这和 slice 完全不同。

---

## 【代码块 6】slice 赋值会共享底层数组

```go
import "fmt"

s1 := []int{1, 2, 3}
s2 := s1

s2[0] = 100

fmt.Println("s1:", s1)
fmt.Println("s2:", s2)
```

---

## 【文本块 10】代码解释

这里 s1 和 s2 都会变成 `[100 2 3]`。

因为 slice 赋值时复制的是 slice header，而不是底层数组。

s1 和 s2 的 header 是两份，但 header 里的 data pointer 指向同一个底层数组，所以修改元素会互相影响。

---

# 四、slice 底层结构

## 【文本块 11】Q：slice 的底层结构是什么？

slice 的底层可以理解成一个结构体，包含三个字段：

```go
type slice struct {
    array unsafe.Pointer
    len   int
    cap   int
}
```

这不是 Go 代码里直接暴露给我们的类型，而是 runtime 层面的抽象。

array 指向底层数组的起始位置。
len 表示当前 slice 能访问到的元素数量。
cap 表示从 slice 起始位置到底层数组末尾还能容纳多少元素。

所以 slice 本身很小，通常就是三个机器字大小。真正的数据存在底层数组里。

面试里最重要的是讲清楚：

> slice 是一个“描述符”，不是数据本身。复制 slice 只会复制描述符，不会复制底层数组。

---

## 【代码块 7】观察 len 和 cap

```go
import "fmt"

s := make([]int, 3, 5)

fmt.Println("s:", s)
fmt.Println("len:", len(s))
fmt.Println("cap:", cap(s))
```

---

## 【文本块 12】len 和 cap 的区别

len 表示当前 slice 里已经能访问的元素个数。

cap 表示在不扩容的前提下，当前 slice 最多能扩展到多长。

例如：

```go
s := make([]int, 3, 5)
```

此时 len 是 3，cap 是 5。

你可以访问 `s[0]`、`s[1]`、`s[2]`，但不能访问 `s[3]`，因为它超过了 len。

不过可以 append 两次而不触发扩容，因为容量还有剩余。

---

## 【代码块 8】len 之外不能直接访问，但可以 append

```go
import "fmt"

s := make([]int, 3, 5)

s[0] = 1
s[1] = 2
s[2] = 3

// 下面这行会 panic：index out of range
// s[3] = 4

s = append(s, 4)
s = append(s, 5)

fmt.Println(s)
fmt.Println("len:", len(s), "cap:", cap(s))
```

---

# 五、slice 扩容机制

## 【文本块 13】Q：slice append 时扩容机制是什么？

append 的基本逻辑是：

如果当前 slice 的 len 小于 cap，说明底层数组还有空间，append 会直接把新元素放到原底层数组里。

如果 len 已经等于 cap，说明底层数组放不下了，runtime 会分配一个新的、更大的底层数组，把旧数据复制过去，再把新元素追加进去，最后返回一个指向新数组的 slice。

所以 append 一定要接收返回值：

```go
s = append(s, x)
```

不能只写：

```go
append(s, x)
```

因为扩容后，新的 slice header 可能已经指向新的底层数组了。

关于扩容倍率，面试里不用死背每个版本的源码细节，但要知道大方向：

早期小容量时，扩容通常比较激进，接近翻倍。
容量变大后，扩容倍率会下降，以减少内存浪费。
最终新容量还会根据元素大小和内存分配规格做对齐。

所以比较稳的回答是：

> slice 扩容不是永远 2 倍。小容量时通常近似翻倍，容量较大后增长比例会降低，并且 runtime 会结合元素大小、内存分配规格进行调整。因此不要在面试里机械说 Go slice 永远 2 倍扩容。

---

## 【代码块 9】观察 slice 扩容

```go
import "fmt"

s := make([]int, 0)

for i := 0; i < 20; i++ {
    oldCap := cap(s)
    s = append(s, i)
    if cap(s) != oldCap {
        fmt.Printf("append %2d: len=%2d cap=%2d\n", i, len(s), cap(s))
    }
}
```

---

## 【文本块 14】追问：append 为什么必须接收返回值？

因为 append 可能返回一个新的 slice header。

如果没有扩容，返回的新 slice 和旧 slice 指向同一个底层数组，只是 len 变大。

如果发生扩容，返回的新 slice 会指向新的底层数组。旧 slice 仍然指向旧数组。

所以必须写：

```go
s = append(s, x)
```

这是 Go 里非常重要的工程习惯。

---

## 【代码块 10】append 后是否影响原 slice：未扩容场景

```go
import "fmt"

s1 := make([]int, 2, 4)
s1[0] = 1
s1[1] = 2

s2 := s1

s2 = append(s2, 3)
s2[0] = 100

fmt.Println("s1:", s1, "len:", len(s1), "cap:", cap(s1))
fmt.Println("s2:", s2, "len:", len(s2), "cap:", cap(s2))
```

---

## 【文本块 15】代码解释

这里 s1 和 s2 一开始共享底层数组。

因为 cap 还有空间，s2 append 时没有扩容，所以 s2 仍然和 s1 共享底层数组。

因此 `s2[0] = 100` 会影响 s1。

但是注意，s1 的 len 仍然是 2，所以它看不到第三个元素 3；但底层数组里其实已经写进去了。

---

## 【代码块 11】append 后是否影响原 slice：扩容场景

```go
import "fmt"

s1 := make([]int, 2, 2)
s1[0] = 1
s1[1] = 2

s2 := s1

s2 = append(s2, 3)
s2[0] = 100

fmt.Println("s1:", s1, "len:", len(s1), "cap:", cap(s1))
fmt.Println("s2:", s2, "len:", len(s2), "cap:", cap(s2))
```

---

## 【文本块 16】代码解释

这里 s1 的 len 和 cap 都是 2，已经没有剩余容量。

s2 append 时会触发扩容，runtime 会创建新的底层数组。

因此 s2 修改第一个元素不会影响 s1。

这就是 slice 面试里最常考的一句话：

> slice 共享底层数组时，修改元素会互相影响；append 是否互相影响，取决于是否触发扩容。

---

# 六、slice 截取与底层数组共享

## 【文本块 17】Q：slice 截取会发生拷贝吗？

不会。

例如：

```go
s2 := s1[1:3]
```

这只是创建了一个新的 slice header，让它指向原底层数组的一段区域。

所以 s1 和 s2 仍然共享底层数组。

这意味着两个风险：

第一，修改子 slice 会影响原 slice。

第二，如果从一个很大的 slice 中截取出很小一段，只要这个小 slice 还活着，整个大数组都可能无法被 GC 回收，导致内存占用异常。

第二个问题就是 Go 里常说的“slice 截取导致的内存泄漏”或“内存滞留”。

---

## 【代码块 12】截取后共享底层数组

```go
import "fmt"

s1 := []int{1, 2, 3, 4, 5}
s2 := s1[1:4]

s2[0] = 200

fmt.Println("s1:", s1)
fmt.Println("s2:", s2)
```

---

## 【文本块 18】追问：如何避免截取大 slice 导致内存滞留？

如果你只需要保留小片段，并希望大数组能被 GC 回收，可以用 copy 创建一份新的小数组。

常见写法：

```go
small := make([]byte, len(part))
copy(small, part)
```

或者：

```go
small := append([]byte(nil), part...)
```

这样 small 会拥有独立的底层数组，不再引用原来的大数组。

---

## 【代码块 13】用 copy 解除底层数组引用

```go
import "fmt"

big := []byte("abcdefghijklmnopqrstuvwxyz")

part := big[0:3]

small := make([]byte, len(part))
copy(small, part)

big[0] = 'X'

fmt.Println("part:", string(part))
fmt.Println("small:", string(small))
```

---

## 【文本块 19】代码解释

part 仍然引用 big 的底层数组，所以 big 修改后，part 也会变化。

small 是 copy 出来的独立数组，所以不会受 big 影响。

在工程中，如果从大文件、大响应体、大日志 buffer 中截取一小段长期保存，一定要考虑 copy，否则很容易出现“明明只存了几 KB，内存却一直下不来”的问题。

---

# 七、slice 删除元素

## 【文本块 20】Q：Go 里怎么删除 slice 元素？

Go 没有内置的 slice delete 函数，常见写法是使用 append 拼接前后两段。

删除下标 i 的元素：

```go
s = append(s[:i], s[i+1:]...)
```

这个写法会复用原底层数组，时间复杂度是 O(n)，因为需要移动后面的元素。

如果不要求保持顺序，可以把最后一个元素覆盖到要删除的位置，然后截断最后一个元素：

```go
s[i] = s[len(s)-1]
s = s[:len(s)-1]
```

这种方式是 O(1)，但会改变元素顺序。

如果 slice 里保存的是指针或大对象引用，删除后还要注意把不再使用的位置置 nil，帮助 GC 回收。

---

## 【代码块 14】保持顺序删除

```go
import "fmt"

s := []int{1, 2, 3, 4, 5}

i := 2
s = append(s[:i], s[i+1:]...)

fmt.Println(s)
```

---

## 【代码块 15】不保持顺序删除

```go
import "fmt"

s := []int{1, 2, 3, 4, 5}

i := 2
s[i] = s[len(s)-1]
s = s[:len(s)-1]

fmt.Println(s)
```

---

## 【文本块 21】追问：删除指针元素时为什么要置 nil？

假设 slice 里存的是指针：

```go
s := []*User{...}
```

如果你删除元素后，底层数组的某个位置仍然残留着被删除对象的指针，即使 len 变短了，只要底层数组还活着，这个对象仍然可能无法被 GC 回收。

所以更严谨的写法是：

```go
copy(s[i:], s[i+1:])
s[len(s)-1] = nil
s = s[:len(s)-1]
```

这在长生命周期 slice 中尤其重要。

---

# 八、nil slice 和 empty slice

## 【文本块 22】Q：nil slice 和 empty slice 有什么区别？

nil slice：

```go
var s []int
```

它的值是 nil，len 和 cap 都是 0。

empty slice：

```go
s := []int{}
```

或者：

```go
s := make([]int, 0)
```

它不是 nil，但 len 和 cap 也是 0。

二者大多数情况下都可以正常 append，也都可以 range。

区别主要体现在：

* 是否等于 nil
* JSON 序列化结果可能不同
* API 语义不同

在 encoding/json 中，nil slice 通常会被编码成 null，empty slice 会被编码成 []。

工程里，如果你希望返回给前端的是空数组而不是 null，通常要初始化为空 slice。

---

## 【代码块 16】nil slice 和 empty slice

```go
import (
    "encoding/json"
    "fmt"
)

var s1 []int
s2 := []int{}

b1, _ := json.Marshal(s1)
b2, _ := json.Marshal(s2)

fmt.Println(s1 == nil, string(b1))
fmt.Println(s2 == nil, string(b2))
```

---

## 【文本块 23】工程建议

对内部逻辑来说，nil slice 和 empty slice 差异不大，因为都可以 len、range、append。

但对接口返回来说，建议保持稳定语义。

如果 API 文档说某字段是数组，最好返回 `[]`，不要有时返回 `null`。

例如：

```go
users := make([]User, 0)
```

比：

```go
var users []User
```

在对外响应里更稳定。

---

# 九、map 基础

## 【文本块 24】Q：Go map 的底层结构是什么？

Go map 是哈希表。

从使用层面看，map 是 key-value 结构：

```go
m := map[string]int{
    "a": 1,
    "b": 2,
}
```

从底层看，map 的核心结构是 runtime.hmap。

hmap 里会维护：

* 元素数量 count
* bucket 数量相关信息
* hash seed
* 指向 bucket 数组的指针
* 扩容时旧 bucket 数组的指针
* 渐进迁移状态

bucket 是 map 的基本存储单元。每个 bucket 里可以存多个 key/value。发生哈希冲突时，如果一个 bucket 放不下，会挂 overflow bucket。

所以 Go map 的大致定位流程是：

1. 对 key 计算 hash
2. 根据 hash 的低位定位到 bucket
3. 在 bucket 内比较 tophash 和 key
4. 找到则返回 value
5. 找不到则继续查 overflow bucket

面试里不需要把 runtime 源码逐行背出来，但要能讲清楚：

> Go map 是哈希表，底层由 hmap 管理元信息，由 bucket 存储 key/value。hash 后先定位 bucket，再在 bucket 内查找；冲突严重时会使用 overflow bucket。

---

## 【代码块 17】map 基本使用

```go
import "fmt"

m := make(map[string]int)

m["apple"] = 10
m["banana"] = 20

fmt.Println(m["apple"])
fmt.Println(m["orange"])
```

---

## 【文本块 25】追问：访问不存在的 key 返回什么？

访问不存在的 key，会返回 value 类型的零值。

例如 map[string]int 中不存在的 key 返回 0。
map[string]string 中不存在的 key 返回空字符串。
map[string]*User 中不存在的 key 返回 nil。

所以如果 value 类型的零值本身也是合法值，就需要用 comma ok 语法区分：

```go
v, ok := m[key]
```

ok 为 true 表示 key 存在。
ok 为 false 表示 key 不存在。

---

## 【代码块 18】comma ok 判断 key 是否存在

```go
import "fmt"

m := map[string]int{
    "apple": 0,
}

v1, ok1 := m["apple"]
v2, ok2 := m["banana"]

fmt.Println("apple:", v1, ok1)
fmt.Println("banana:", v2, ok2)
```

---

## 【文本块 26】代码解释

apple 的值是 0，但 ok 是 true，说明 key 存在。
banana 的值也是 0，但 ok 是 false，说明 key 不存在。

所以不能只通过 value 是否为零值来判断 key 是否存在。

---

# 十、map 的 key 有什么要求

## 【文本块 27】Q：Go map 的 key 有什么要求？

map 的 key 必须是可比较类型，也就是支持 `==` 和 `!=` 的类型。

可以作为 key 的常见类型：

* bool
* int、float、string
* pointer
* channel
* interface，前提是动态值可比较
* array，前提是元素可比较
* struct，前提是所有字段都可比较

不能作为 key 的常见类型：

* slice
* map
* function

因为这些类型不可比较。

为什么 map key 必须可比较？

因为哈希表查找时，hash 只能定位到候选 bucket，最终还需要判断两个 key 是否真的相等。如果 key 不能比较，就无法确认是不是同一个 key。

---

## 【代码块 19】struct 作为 map key

```go
import "fmt"

type Point struct {
    X int
    Y int
}

m := make(map[Point]string)

m[Point{X: 1, Y: 2}] = "A"

fmt.Println(m[Point{X: 1, Y: 2}])
```

---

## 【文本块 28】追问：为什么 slice 不能作为 map key？

因为 slice 不支持相等比较。

slice 是一个描述符，里面包含指针、len、cap。即使两个 slice 内容一样，它们也可能指向不同的底层数组；即使指向同一个底层数组，也可能有不同的 len 和 cap。更重要的是，slice 内容可变，如果允许作为 map key，插入后内容变化会破坏哈希表的定位逻辑。

所以 Go 直接规定 slice 不可比较，不能作为 map key。

如果业务上确实想用一段序列作为 key，常见做法是转换成 string，例如：

```go
key := string(bytes)
```

或者自己构造稳定的序列化 key。

---

# 十一、map 遍历无序

## 【文本块 29】Q：为什么 Go map 遍历是无序的？

Go map 遍历顺序是故意随机化的。

原因主要有两个：

第一，map 本身是哈希表，元素的存储位置和 hash、bucket、扩容状态有关，本来就不应该依赖顺序。

第二，Go runtime 故意让遍历起点带有随机性，防止开发者无意中依赖 map 的遍历顺序。否则代码在小数据量或某个版本里看起来稳定，一旦升级版本、扩容或者换环境就可能出 bug。

所以工程里如果需要有序输出，必须自己排序 key。

---

## 【代码块 20】map 遍历顺序不要依赖

```go
import "fmt"

m := map[string]int{
    "a": 1,
    "b": 2,
    "c": 3,
    "d": 4,
}

for k, v := range m {
    fmt.Println(k, v)
}
```

---

## 【代码块 21】需要有序时先排序 key

```go
import (
    "fmt"
    "sort"
)

m := map[string]int{
    "banana": 2,
    "apple":  1,
    "orange": 3,
}

keys := make([]string, 0, len(m))
for k := range m {
    keys = append(keys, k)
}

sort.Strings(keys)

for _, k := range keys {
    fmt.Println(k, m[k])
}
```

---

# 十二、map 扩容

## 【文本块 30】Q：Go map 是怎么扩容的？

Go map 扩容通常有两类触发原因：

第一，负载因子过高。
元素越来越多，bucket 越来越满，查找和插入效率下降，需要扩容。

第二，overflow bucket 太多。
即使元素总数没有特别夸张，如果哈希冲突多，overflow bucket 很多，也会影响性能，需要整理。

Go map 的扩容不是一次性把所有旧 bucket 搬到新 bucket。

它采用渐进式迁移。

也就是说，扩容开始后，hmap 会同时保留旧 bucket 数组和新 bucket 数组。后续每次读写 map 时，runtime 会顺便迁移一部分旧 bucket 到新 bucket。

这样可以避免一次性搬迁大量数据导致长时间卡顿。

面试里可以这样回答：

> Go map 扩容采用渐进式迁移，不会一次性搬完所有 bucket。扩容期间新旧 bucket 会同时存在，后续 map 操作会逐步完成迁移，从而把扩容成本摊到多次操作中。

---

## 【文本块 31】追问：map 扩容时，原来的 key 位置一定会变吗？

不一定。

扩容后 bucket 数量变化，hash 的更多位会参与 bucket 定位。一部分 key 仍然留在原相对位置，另一部分 key 会迁移到新的位置。

这和 Java HashMap 扩容时“原索引或原索引 + oldCap”的思想有点相似，都是利用 hash 位来决定迁移方向。

但是 Go map 的实现细节和 Java HashMap 不一样，不能直接混为一谈。

---

# 十三、map 并发安全

## 【文本块 32】Q：Go map 是线程安全的吗？

不是。

Go 原生 map 不支持并发读写。

如果一个 goroutine 正在写 map，另一个 goroutine 同时读或写，就可能触发 fatal error：

```text
fatal error: concurrent map read and map write
```

注意，这不是普通 panic，而是 runtime fatal error，通常不能通过 recover 当作正常业务异常处理。

为什么 map 不做成默认线程安全？

因为大多数 map 使用场景都是单 goroutine 或者外部已经有同步控制。如果每个 map 操作都内置锁，会让所有场景都付出额外成本。Go 的设计倾向是把并发控制交给开发者显式处理。

常见解决方案有三种：

第一，map + sync.Mutex。
适合读写都比较多，逻辑需要精确控制的场景。

第二，map + sync.RWMutex。
适合读多写少的场景。

第三，sync.Map。
适合读多写少、key 相对稳定、写入较少的场景，例如缓存、单例注册表、连接状态表等。

---

## 【代码块 22】map + Mutex

```go
import (
    "fmt"
    "sync"
)

type SafeMap struct {
    mu sync.Mutex
    m  map[string]int
}

func NewSafeMap() *SafeMap {
    return &SafeMap{
        m: make(map[string]int),
    }
}

func (s *SafeMap) Set(key string, value int) {
    s.mu.Lock()
    defer s.mu.Unlock()
    s.m[key] = value
}

func (s *SafeMap) Get(key string) (int, bool) {
    s.mu.Lock()
    defer s.mu.Unlock()
    v, ok := s.m[key]
    return v, ok
}

sm := NewSafeMap()
sm.Set("a", 1)
v, ok := sm.Get("a")

fmt.Println(v, ok)
```

---

## 【代码块 23】map + RWMutex

```go
import (
    "fmt"
    "sync"
)

type SafeRWMap struct {
    mu sync.RWMutex
    m  map[string]int
}

func NewSafeRWMap() *SafeRWMap {
    return &SafeRWMap{
        m: make(map[string]int),
    }
}

func (s *SafeRWMap) Set(key string, value int) {
    s.mu.Lock()
    defer s.mu.Unlock()
    s.m[key] = value
}

func (s *SafeRWMap) Get(key string) (int, bool) {
    s.mu.RLock()
    defer s.mu.RUnlock()
    v, ok := s.m[key]
    return v, ok
}

sm := NewSafeRWMap()
sm.Set("a", 1)
v, ok := sm.Get("a")

fmt.Println(v, ok)
```

---

## 【文本块 33】追问：RWMutex 一定比 Mutex 快吗？

不一定。

RWMutex 的优势在于多个读操作可以并发执行，适合读多写少。

但如果写操作频繁，RWMutex 的读写协调成本可能更高，甚至不如普通 Mutex。

工程里不要机械地认为 RWMutex 一定更高级。要看读写比例、临界区耗时、竞争程度，最好用 benchmark 或压测验证。

---

# 十四、sync.Map

## 【文本块 34】Q：sync.Map 的底层思想是什么？

sync.Map 是 Go 标准库提供的并发安全 map。

它不是简单的 map 加一把大锁，而是针对特定场景做了优化。

sync.Map 内部大致有两层：

第一层是 read map。
主要用于无锁读取，读性能很高。

第二层是 dirty map。
保存新增或被修改的数据，需要加锁访问。

当读取 key 时，先查 read map。
如果 read map 找不到，再加锁查 dirty map。
如果 miss 次数达到一定阈值，dirty map 会提升为新的 read map。

这个设计非常适合读多写少、key 相对稳定的场景。

但是如果你的场景是频繁写入、频繁删除、key 变化很大，sync.Map 不一定比 map + Mutex 更好。

面试里可以这样回答：

> sync.Map 适合读多写少、key 稳定的场景。它通过 read/dirty 双 map 减少读路径加锁，但在写多或者 key 频繁变化的场景下，性能未必优于 map + Mutex。

---

## 【代码块 24】sync.Map 基本使用

```go
import (
    "fmt"
    "sync"
)

var m sync.Map

m.Store("a", 1)
m.Store("b", 2)

v, ok := m.Load("a")
fmt.Println(v, ok)

m.Delete("b")

m.Range(func(key, value any) bool {
    fmt.Println(key, value)
    return true
})
```

---

## 【文本块 35】追问：sync.Map 和 map + Mutex 怎么选？

我的工程回答一般是：

如果是普通业务 map，读写逻辑明确，优先用 map + Mutex 或 RWMutex。因为类型安全、逻辑清楚、性能也可控。

如果是读多写少、key 基本稳定的全局缓存、注册表、连接映射，可以考虑 sync.Map。

如果需要复合操作，例如“先查再改再写”，sync.Map 不一定方便，因为 Load 和 Store 是单次操作安全，不代表你的业务组合逻辑天然原子。

例如：

```go
v, _ := m.Load(key)
m.Store(key, v+1)
```

这不是原子递增。并发下仍然可能丢更新。

---

# 十五、map 使用工程陷阱

## 【文本块 36】陷阱 1：map value 是 struct 时，不能直接修改字段

如果 map 的 value 是 struct，下面这种写法是不允许的：

```go
m["a"].Count++
```

原因是 map 取出来的 value 不是可寻址的变量。runtime 可能因为扩容移动 value 的位置，所以不能直接拿它的字段地址做修改。

解决方式有两种：

第一，取出来，修改后再放回去。
第二，把 value 设计成指针。

---

## 【代码块 25】修改 map 中的 struct value

```go
import "fmt"

type Counter struct {
    Count int
}

m := map[string]Counter{
    "a": {Count: 1},
}

// m["a"].Count++ // 这行不能编译

c := m["a"]
c.Count++
m["a"] = c

fmt.Println(m["a"])
```

---

## 【代码块 26】map value 使用指针

```go
import "fmt"

type Counter struct {
    Count int
}

m := map[string]*Counter{
    "a": {Count: 1},
}

m["a"].Count++

fmt.Println(m["a"].Count)
```

---

## 【文本块 37】陷阱 2：遍历 map 时可以删除吗？

在 Go 中，遍历 map 的过程中删除当前 map 的 key 是允许的。

例如：

```go
for k := range m {
    delete(m, k)
}
```

这是安全的。

但是遍历过程中新增 key，是否会在本轮遍历中被看到是不确定的，不应该依赖这个行为。

工程建议：

* 遍历中删除通常可以接受
* 遍历中新增不要依赖结果
* 如果要稳定处理，先收集 key，再统一操作

---

# 十六、string 底层结构

## 【文本块 38】Q：Go string 的底层是什么？

Go 的 string 本质上是一个只读的字节序列。

底层可以抽象理解为：

```go
type stringStruct struct {
    str unsafe.Pointer
    len int
}
```

str 指向底层字节数据。
len 表示字节长度。

注意，len 是字节数，不是字符数。

Go 的 string 默认通常用 UTF-8 编码保存文本。对于英文字符，一个字符通常是 1 个字节；对于中文字符，一个字符通常是 3 个字节。

所以：

```go
len("hello") == 5
len("你好") == 6
```

因为“你”和“好”各占 3 个字节。

面试里最容易加分的一句话是：

> Go string 是只读字节序列，不是字符数组。len 返回字节数，下标访问返回 byte，range 遍历按 UTF-8 解码后返回 rune。

---

## 【代码块 27】string 的 len 是字节数

```go
import "fmt"

s1 := "hello"
s2 := "你好"

fmt.Println(len(s1))
fmt.Println(len(s2))
```

---

## 【代码块 28】下标访问拿到 byte

```go
import "fmt"

s := "你好"

fmt.Printf("%T %v\n", s[0], s[0])
fmt.Printf("%x\n", s[0])
```

---

## 【文本块 39】代码解释

`s[0]` 拿到的是第一个字节，不是第一个字符。

“你”在 UTF-8 中占 3 个字节，所以单独拿 s[0] 只是它编码中的第一个 byte。

如果要按字符遍历，应该用 range。

---

## 【代码块 29】range string 拿到 rune

```go
import "fmt"

s := "你好"

for i, r := range s {
    fmt.Printf("index=%d, rune=%c, type=%T\n", i, r, r)
}
```

---

## 【文本块 40】range 的 index 为什么不是 0、1？

range 遍历 string 时，index 是每个 rune 起始位置的字节下标。

对于 `"你好"`：

“你”从字节下标 0 开始。
“好”从字节下标 3 开始。

所以输出的 index 通常是 0 和 3，而不是 0 和 1。

这点在处理字符串截取、字符计数时非常重要。

---

# 十七、byte 和 rune

## 【文本块 41】Q：byte 和 rune 有什么区别？

byte 是 uint8 的别名，表示一个字节。

rune 是 int32 的别名，表示一个 Unicode code point。

如果处理的是二进制数据、网络数据、文件内容、加密压缩等，通常用 []byte。

如果处理的是字符语义，比如统计字符数、按字符反转字符串、判断中文字符，通常用 []rune。

注意，rune 不等于“用户眼中的一个字符”。某些复杂 emoji 可能由多个 code point 组合而成。但在大多数后端面试和普通中文英文处理场景里，把 rune 理解成 Unicode 字符已经足够。

---

## 【代码块 30】[]byte 和 []rune 的区别

```go
import "fmt"

s := "Go语言"

bs := []byte(s)
rs := []rune(s)

fmt.Println("bytes len:", len(bs), bs)
fmt.Println("runes len:", len(rs), rs)
```

---

## 【文本块 42】追问：如何正确反转包含中文的字符串？

如果直接按 byte 反转，会破坏 UTF-8 编码。

应该先转成 []rune，再反转。

---

## 【代码块 31】反转中文字符串

```go
import "fmt"

func reverseString(s string) string {
    r := []rune(s)
    for i, j := 0, len(r)-1; i < j; i, j = i+1, j-1 {
        r[i], r[j] = r[j], r[i]
    }
    return string(r)
}

fmt.Println(reverseString("Go语言"))
```

---

# 十八、string 不可变

## 【文本块 43】Q：Go string 是不可变的吗？

是的。

Go string 一旦创建，其内容不能被修改。

例如下面这种写法不能编译：

```go
s := "hello"
s[0] = 'H'
```

为什么 string 要设计成不可变？

第一，安全。
字符串经常作为 map key、配置项、文件路径、URL、SQL 片段等，如果可变，会引发很多隐蔽问题。

第二，便于共享。
多个 string 可以安全共享同一段底层字节数据，不用担心某一方修改内容。

第三，利于编译器和 runtime 优化。
不可变语义让字符串在传递、切片、比较、作为 map key 时更容易保持稳定行为。

不过要注意：

不可变的是 string 内容，不是变量绑定。

你可以让变量 s 指向另一个新字符串：

```go
s = "world"
```

但不能修改原字符串内部的某个字节。

---

## 【代码块 32】string 变量可以重新赋值

```go
import "fmt"

s := "hello"
s = "world"

fmt.Println(s)
```

---

## 【文本块 44】追问：string 截取会复制底层数据吗？

在很多场景下，字符串截取可能共享底层数据，具体实现会受编译器和 runtime 优化影响。

从工程角度看，要注意一个类似 slice 的问题：

如果你从一个很大的字符串里截取一个很小的子串，并长期保存这个子串，可能导致原大字符串的底层数据无法及时释放。

如果你明确希望复制一份小字符串，可以通过构造新字符串来避免长期引用大对象。

常见方式：

```go
small := string([]byte(sub))
```

不过这类优化不要滥用。只有在处理大字符串、长期持有小片段、内存确实异常时再考虑。

---

# 十九、string 和 []byte 转换

## 【文本块 45】Q：string 和 []byte 转换会发生拷贝吗？

通常情况下，string 和 []byte 相互转换会发生拷贝。

原因是二者语义不同：

string 是不可变的。
[]byte 是可变的。

如果 string 转 []byte 不拷贝，那么修改 []byte 就可能破坏 string 的不可变性。

如果 []byte 转 string 不拷贝，那么后续修改 []byte 也可能影响 string。

所以为了保证语言语义安全，正常转换通常会复制底层数据。

示例：

```go
b := []byte(s)
s2 := string(b)
```

这两个转换一般都意味着分配和拷贝。

编译器在某些临时场景下可能做优化，但面试和工程上应默认认为会有成本。

---

## 【代码块 33】string 转 []byte 后修改不影响原 string

```go
import "fmt"

s := "hello"
b := []byte(s)

b[0] = 'H'

fmt.Println(s)
fmt.Println(string(b))
```

---

## 【文本块 46】工程建议

频繁做 string 和 []byte 转换会带来额外内存分配和拷贝。

典型场景：

* HTTP 请求体读取
* JSON 编解码
* 日志拼接
* Redis value 序列化
* 文件内容处理

如果底层 API 支持 []byte，就尽量使用 []byte。
如果是文本处理、map key、不可变标识，就使用 string。

不要为了“看起来方便”反复转换。

---

# 二十、字符串拼接

## 【文本块 47】Q：Go 里怎么高效拼接字符串？

常见方式有几种：

第一，少量字符串拼接，用 `+` 就可以。

```go
s := "hello" + " " + "world"
```

编译器对简单拼接会做优化，没必要过度设计。

第二，循环中大量拼接，推荐 strings.Builder。

因为 string 不可变，如果在循环里反复 `s += part`，每次都可能创建新字符串并拷贝旧内容，复杂度可能退化。

第三，如果处理的是字节数据，也可以用 bytes.Buffer。

strings.Builder 更适合最终生成 string。
bytes.Buffer 更适合处理 []byte 或者需要同时读写 byte 的场景。

---

## 【代码块 34】不推荐：循环里直接 +=

```go
import "fmt"

s := ""

for i := 0; i < 5; i++ {
    s += fmt.Sprintf("%d", i)
}

fmt.Println(s)
```

---

## 【代码块 35】推荐：strings.Builder

```go
import (
    "fmt"
    "strings"
)

var b strings.Builder

for i := 0; i < 5; i++ {
    b.WriteString(fmt.Sprintf("%d", i))
}

s := b.String()

fmt.Println(s)
```

---

## 【文本块 48】追问：strings.Builder 为什么高效？

strings.Builder 内部维护一个可增长的 byte buffer。

写入时，它可以尽量复用内部 buffer，避免每次拼接都生成一个新的 string。

最后调用 String() 时，再得到最终字符串。

如果能预估大小，还可以提前 Grow：

```go
b.Grow(1024)
```

这样可以减少扩容次数。

面试里可以这样回答：

> 少量拼接直接用 +，循环大量拼接用 strings.Builder；如果处理的是字节流或二进制数据，用 bytes.Buffer 更自然。

---

# 二十一、类型别名和自定义类型

## 【文本块 49】Q：type MyInt int 和 type MyInt = int 有什么区别？

Go 里有两个容易混淆的概念：自定义类型和类型别名。

```go
type MyInt int
```

这是定义了一个新类型 MyInt，它的底层类型是 int，但 MyInt 和 int 是不同的类型，不能直接互相赋值，需要显式转换。

```go
type MyInt = int
```

这是类型别名，MyInt 和 int 完全是同一个类型，只是多了一个名字。

自定义类型常用于增强语义，例如：

```go
type UserID int64
type OrderID int64
```

这样 UserID 和 OrderID 虽然底层都是 int64，但类型上不同，可以避免误传。

---

## 【代码块 36】自定义类型需要显式转换

```go
import "fmt"

type UserID int64
type OrderID int64

var uid UserID = 100
var oid OrderID = 200

fmt.Println(uid, oid)

// uid = oid // 不能直接赋值

uid = UserID(oid)

fmt.Println(uid)
```

---

## 【文本块 50】工程建议

在业务代码里，不要所有 ID 都用 int64 或 string 裸奔。

例如：

```go
type UserID int64
type ProductID int64
type OrderID int64
```

这样可以让编译器帮你发现一部分参数误传问题。

这就是 Go 类型系统在工程实践中的价值：不是为了复杂，而是为了让错误更早暴露。

---

# 二十二、struct 值语义与指针语义

## 【文本块 51】Q：struct 传值还是传指针？

这要看场景。

传值的优点：

* 不容易被函数内部修改
* 没有 nil 问题
* 对小结构体很简单
* 值语义更清晰

传指针的优点：

* 避免复制大结构体
* 可以在函数内部修改原对象
* 方法需要改变接收者状态时必须用指针接收者
* 可以表达可选值或缺失值

工程里常见规则：

小的、不可变语义的对象，可以传值。
大的结构体，或者需要修改状态，传指针。
含有 sync.Mutex、sync.WaitGroup 等不能复制的字段时，必须传指针，避免复制锁。
方法接收者尽量保持一致，不要一部分值接收者、一部分指针接收者混用。

---

## 【代码块 37】值接收者不会修改原对象

```go
import "fmt"

type User struct {
    Name string
}

func (u User) Rename(name string) {
    u.Name = name
}

u := User{Name: "Tom"}
u.Rename("Jerry")

fmt.Println(u.Name)
```

---

## 【代码块 38】指针接收者可以修改原对象

```go
import "fmt"

type User struct {
    Name string
}

func (u *User) Rename(name string) {
    u.Name = name
}

u := User{Name: "Tom"}
u.Rename("Jerry")

fmt.Println(u.Name)
```

---

## 【文本块 52】追问：为什么含 Mutex 的 struct 不能复制？

sync.Mutex 里有锁状态。

如果一个结构体里包含 Mutex，你把这个结构体复制了一份，就会复制锁状态。这样两个对象的业务数据看起来相关，但锁已经不是同一把锁，可能破坏并发保护语义。

所以包含 Mutex、RWMutex、WaitGroup、Cond 等同步原语的结构体，一般都不能复制，应该用指针传递。

这也是 Go 代码审查里非常重要的点。

---

# 二十三、interface 与 nil

## 【文本块 53】Q：interface 的 nil 问题是什么？

Go 的 interface 底层可以理解成两部分：

* 动态类型
* 动态值

一个 interface 只有在动态类型和动态值都为 nil 时，才等于 nil。

这就会导致一个经典坑：

一个具体类型的 nil 指针赋值给 interface 后，这个 interface 本身不等于 nil，因为它已经有了动态类型。

---

## 【代码块 39】interface nil 陷阱

```go
import "fmt"

type MyError struct{}

func (e *MyError) Error() string {
    return "my error"
}

func getErr() error {
    var e *MyError = nil
    return e
}

err := getErr()

fmt.Println(err == nil)
fmt.Printf("%T %#v\n", err, err)
```

---

## 【文本块 54】代码解释

这里 err 不等于 nil。

因为返回的 error 接口里，动态类型是 `*MyError`，动态值是 nil。

也就是：

```text
(type = *MyError, value = nil)
```

而真正的 nil interface 是：

```text
(type = nil, value = nil)
```

工程里写自定义 error 时，一定要避免返回“带类型的 nil”。

更推荐：

```go
if noError {
    return nil
}
return &MyError{}
```

---

# 二十四、本章面试速记版

## 【文本块 55】Go 类型系统速记

Go 只有值传递，没有引用传递。

slice、map、channel 之所以看起来像引用传递，是因为它们的值内部包含指向底层结构的指针。

new 返回指针，适用于任意类型；make 只用于 slice、map、channel，负责初始化 runtime 底层结构。

array 是定长值类型，长度是类型的一部分；slice 是动态视图，底层是数组，header 包含 pointer、len、cap。

slice append 不一定扩容。不扩容时共享底层数组，扩容后指向新数组。append 必须接收返回值。

slice 截取不会复制底层数组。长期保留大 slice 的小切片，可能造成内存滞留，需要 copy。

nil slice 和 empty slice 大多数操作类似，但 JSON 表现不同，接口返回一般更推荐 empty slice。

map 是哈希表，底层由 hmap 和 bucket 组成。key 必须可比较。map 遍历无序。map 并发读写不安全。

sync.Map 适合读多写少、key 稳定的场景，不是所有并发 map 的默认最优解。

string 是只读字节序列，不是字符数组。len 返回字节数，下标访问返回 byte，range 遍历返回 rune。

byte 是 uint8，表示字节；rune 是 int32，表示 Unicode code point。

string 和 []byte 正常转换通常会拷贝。少量字符串拼接用 +，大量循环拼接用 strings.Builder。

struct 默认值语义。小对象可传值，大对象、需要修改、包含锁的对象应该传指针。

interface 只有动态类型和动态值都为 nil 时，才等于 nil。

---

## 【文本块 56】本章最终面试回答模板

如果面试官让我整体讲 Go 语言基础里最核心的类型，我会这样组织：

Go 的类型系统看起来简单，但核心是理解值语义和引用语义的边界。Go 语言层面所有传参都是值传递，但 slice、map、channel 这类类型的值本身保存了指向底层结构的指针，所以复制以后仍然能影响底层数据。

slice 是对底层数组的描述符，包含 pointer、len、cap。赋值和传参只复制 header，不复制底层数组。append 时如果容量够，会复用原数组；如果容量不够，会扩容并迁移到新数组。所以 append 必须接收返回值，且要注意多个 slice 共享底层数组带来的数据污染和内存滞留问题。

map 是哈希表，底层由 hmap 管理元数据，由 bucket 存储 key/value。key 必须是可比较类型。map 遍历是无序的，原生 map 也不是并发安全的。并发场景下要么用 map 加 Mutex/RWMutex，要么在读多写少、key 稳定的场景下使用 sync.Map。

string 是不可变的字节序列，len 返回字节数，不是字符数。下标访问得到 byte，range 遍历会按 UTF-8 解码得到 rune。处理二进制数据用 []byte，处理字符语义时可以转 []rune。大量字符串拼接应该用 strings.Builder，避免循环里反复创建新字符串。

总结一句话：Go 基础面试不是背语法，而是要讲清楚“值怎么传、底层数据在哪里、什么时候共享、什么时候拷贝、并发下是否安全”。
